# Draftly — Legal Data Processing Pipeline

Turns the **raw case-law + statute corpus** into a clean, structured, linked dataset
for the rule-based retrieval engine. Run top to bottom.

**Stages**
0. Consolidate the case index (all sources -> one table)
1. Load & clean case text
2. Statute citation extraction — the case<->statute graph
3. Assemble per-case records -> `cases.jsonl`
4. Topic tagging & rule tables
5. Retrieval eval set (template) + next steps

Inputs: `data/legal-sources/manifests/*.csv` + the harvested text (gitignored,
rebuildable via `scripts/harvest_*`). Outputs: `data/legal-sources/derived/`
(gitignored). Grounded + deterministic — no LLM/embeddings in this pipeline.

## Setup

In [ ]:
%pip install -q pandas
import json, re, csv
from pathlib import Path
import pandas as pd

# repo root = parent of notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
MAN  = ROOT / 'data/legal-sources/manifests'
TOPICS = ROOT / 'data/legal-sources/topics'
DERIVED = ROOT / 'data/legal-sources/derived'
DERIVED.mkdir(parents=True, exist_ok=True)

# Process a subset first while iterating; set to None for the full run.
SAMPLE_LIMIT = 300
print('ROOT =', ROOT)
print('manifests:', sorted(p.name for p in MAN.glob('*.csv')))

## Stage 0 — Consolidate the case index

Merge the per-judgment manifests (CommonLII reported NLR/SLR + official courts)
into one master table. (Internet Archive is volume-level, handled separately.)
Dedup across sources by citation.

In [ ]:
def load_commonlii():
    f = MAN / 'case-law-commonlii-conveyancing.csv'
    if not f.exists(): return pd.DataFrame()
    d = pd.read_csv(f, dtype=str).fillna('')
    return pd.DataFrame({
        'case_id': 'commonlii-' + d['db'] + '-' + d['year'] + '-' + d['case_no'],
        'source': 'commonlii', 'court': d['db'], 'year': d['year'],
        'citation': d['citation'], 'parties': d['title'],
        'hits': d['hits'], 'text_path': d['text_file'],
    })

def load_courts():
    f = MAN / 'case-law-courts.csv'
    if not f.exists(): return pd.DataFrame()
    d = pd.read_csv(f, dtype=str).fillna('')
    d = d[d['conveyancing'] == '1']
    return pd.DataFrame({
        'case_id': 'courts-' + d['court'] + '-' + d['file'],
        'source': 'courts', 'court': d['court'], 'year': '',
        'citation': '', 'parties': d['file'],
        'hits': d['hits'], 'text_path': d['text_file'],
    })

cases = pd.concat([load_commonlii(), load_courts()], ignore_index=True)
cases['hits'] = pd.to_numeric(cases['hits'], errors='coerce').fillna(0).astype(int)
# dedup: same citation (when present) keeps the highest-hits row
cases = cases.sort_values('hits', ascending=False)
has_cit = cases[cases['citation'].str.strip() != ''].drop_duplicates(subset='citation')
no_cit  = cases[cases['citation'].str.strip() == '']
cases = pd.concat([has_cit, no_cit], ignore_index=True).sort_values(['source', 'court'])
cases.to_csv(DERIVED / 'master_case_index.csv', index=False)
print('total conveyancing cases:', len(cases))
print(cases.groupby(['source', 'court']).size())
cases.head()

## Stage 1 — Load & clean case text

Load a judgment's text and strip common noise (repeated whitespace, page markers).

In [ ]:
def load_text(rel_path: str) -> str:
    if not rel_path: return ''
    p = ROOT / rel_path
    if not p.exists(): p = Path(rel_path)
    return p.read_text(encoding='utf-8', errors='ignore') if p.exists() else ''

def clean_text(t: str) -> str:
    t = re.sub(r'\bPage\s+\d+\s+of\s+\d+\b', ' ', t, flags=re.I)
    t = re.sub(r'\s+', ' ', t)
    return t.strip()

demo = next((r for _, r in cases.iterrows() if load_text(r['text_path'])), None)
if demo is not None:
    txt = clean_text(load_text(demo['text_path']))
    print(demo['case_id'], '|', (demo['citation'] or demo['parties'])[:60])
    print('chars:', len(txt))
    print(txt[:400])

## Stage 2 — Statute citation extraction (the case<->statute graph)

Build a matcher from `source-registry.csv` (statute core names) and scan each case
for the statutes it cites + any `section N` it mentions. Deterministic regex — this
is the graph layer, no LLM.

In [ ]:
reg = pd.read_csv(MAN / 'source-registry.csv', dtype=str).fillna('')

def core_name(title: str) -> str:
    t = re.sub(r'\(amendment\)', '', title, flags=re.I)
    t = re.sub(r'\b(Act|Ordinance|Law|Statute)\b.*$', '', t, flags=re.I)
    t = re.sub(r'\bNo\.?\s*\d+.*$', '', t, flags=re.I)
    return re.sub(r'\s+', ' ', t).strip()

matchers = {}
for _, r in reg.iterrows():
    name = core_name(r['official_title'])
    if len(name) >= 6:
        matchers[r['source_id']] = (name, re.compile(re.escape(name), re.I))
print('statute matchers:', len(matchers))

SEC = re.compile(r'\b(?:section|sec\.?|s\.)\s*(\d+[A-Za-z]?)', re.I)

def extract_links(text: str):
    cited = [sid for sid, (nm, rx) in matchers.items() if rx.search(text)]
    sections = sorted(set(SEC.findall(text)))
    return cited, sections

rows = cases.head(SAMPLE_LIMIT) if SAMPLE_LIMIT else cases
records, edges = [], []
for _, r in rows.iterrows():
    txt = clean_text(load_text(r['text_path']))
    if not txt: continue
    cited, sections = extract_links(txt)
    records.append((r['case_id'], cited, sections, len(txt)))
    for sid in cited:
        edges.append({'case_id': r['case_id'], 'source_id': sid})

edges_df = pd.DataFrame(edges)
edges_df.to_csv(DERIVED / 'case_statute_links.csv', index=False)
print('cases scanned:', len(records), '| case-statute edges:', len(edges_df))
if len(edges_df):
    print('\ntop cited statutes:')
    for sid, n in edges_df['source_id'].value_counts().head(10).items():
        print(f'  {sid}: {n}  ({matchers.get(sid, ("?",))[0]})')

## Stage 3 — Assemble per-case records -> `cases.jsonl`

One JSON record per case: id, source, court, citation, parties, statutes cited,
sections mentioned, text path. This JSONL is what retrieval + eval run on.

In [ ]:
rec_by_id = {cid: (cited, sections, n) for cid, cited, sections, n in records}
out = DERIVED / 'cases.jsonl'
with out.open('w', encoding='utf-8') as fh:
    for _, r in rows.iterrows():
        cited, sections, n = rec_by_id.get(r['case_id'], ([], [], 0))
        if not n: continue
        fh.write(json.dumps({
            'case_id': r['case_id'], 'source': r['source'], 'court': r['court'],
            'year': r['year'], 'citation': r['citation'], 'parties': r['parties'],
            'statutes_cited': cited, 'sections_mentioned': sections,
            'chars': n, 'text_path': r['text_path'],
        }, ensure_ascii=False) + '\n')
print('wrote', out, '(', sum(1 for _ in out.open(encoding='utf-8')), 'records )')

## Stage 4 — Topic tagging & rule tables

Read the 20 topic manifests -> `topic -> source_ids`. Then tag each case to topics
via the statutes it cites (`case -> topics`). Save the rule tables.

In [ ]:
topic_sources = {}
for man in sorted(TOPICS.glob('*/manifest.md')):
    ids = sorted(set(re.findall(r'SRC\d+', man.read_text(encoding='utf-8', errors='ignore'))))
    if ids: topic_sources[man.parent.name] = ids
print('topics with sources:', len(topic_sources))

src_to_topics = {}
for topic, ids in topic_sources.items():
    for sid in ids: src_to_topics.setdefault(sid, set()).add(topic)

case_topics = []
for cid, cited, sections, n in records:
    topics = sorted({t for sid in cited for t in src_to_topics.get(sid, [])})
    case_topics.append({'case_id': cid, 'topics': ';'.join(topics)})
pd.DataFrame(case_topics).to_csv(DERIVED / 'case_topics.csv', index=False)
with (DERIVED / 'topic_sources.json').open('w', encoding='utf-8') as fh:
    json.dump(topic_sources, fh, indent=2)
print('wrote case_topics.csv + topic_sources.json')

## Stage 5 — Retrieval eval set (template) + next steps

Hand-author ~30-50 conveyancing questions with the *expected* statute sections +
cases (gold answers) to measure retrieval (recall@k). Seed template below.

**Next steps beyond this notebook**
- Refine citation -> *section-level* linking (associate `section N` with the nearest
  statute mention).
- Structure the statutes into `## Section N` nodes (Workstream B, one topic first).
- OCR the ~70 scanned Court of Appeal judgments, then re-run Stage 1-3 over them.
- Build the rule-based lookup on top of `cases.jsonl` + `case_statute_links.csv`.

In [ ]:
eval_seed = [
    {'question': 'What are the requisites for a valid deed?', 'expected_sources': 'SRC001', 'expected_sections': '2', 'topics': '05'},
    {'question': 'How is title acquired by prescription?', 'expected_sources': '', 'expected_sections': '', 'topics': '09'},
    {'question': 'What must a notary certify about identity?', 'expected_sources': '', 'expected_sections': '', 'topics': '08'},
]
eval_path = DERIVED / 'retrieval_eval_seed.csv'
pd.DataFrame(eval_seed).to_csv(eval_path, index=False)
print('seed eval set ->', eval_path, '(expand to ~30-50 by hand)')